# Class 1 lab: your first model

**CML3112, Civil Engineering and Architecture**  |  Week 8  |  KLAi at IUEA

In the lab you build three models that predict **cooling load** (in **kW**) from two readings, and you compare them fairly.

**How to use this notebook**

1. Run the cells in order from the top. Press Shift and Enter to run a cell.
2. Every step has the same rhythm: **PREDICT** what will happen, **RUN** the cell, **READ** the output, then **CHANGE** one thing and see what moves.
3. Type your answers in the lines that start with `>`. Double-click a text cell to edit it.
4. If you get an error, read the last line first. The slide called "Four errors you will meet" lists the common ones.

**The words** (the real industry terms, also on the slides): a *feature* is an input column (also called an input variable or predictor). The *target variable* is the column we predict (when it is a class, such as yes or no, it is called a *label*; in Class 2 we make a yes or no label from the target). A *baseline model* is the simplest possible model: it predicts the mean of the training targets for every row, and every other model must beat it. *Prediction error* (also called the *residual*) is actual minus predicted: e = y - y-hat, where y is the actual target and y-hat is the prediction. *MAE* (mean absolute error) is the mean of the absolute errors. *RMSE* (root mean squared error) is the square root of the mean of the squared errors. The *training set* is the group of rows the model learns from. The *validation set* is held-out data: rows we keep back while the model trains, so that we can evaluate it fairly. A *missing value* is an empty cell (NaN in pandas), and *imputation* fills it. *Parameters* are the numbers a model learns (the intercept and coefficients). A *hyperparameter* is a setting we choose before training, such as `max_depth`.

---
## Step 0: Open the data

First we load the file. Upload `engineering_data.csv` in the Files panel on the left of Colab (the folder icon) if you have not already.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# read the file into a table. The raw table is never changed: we clean a copy.
raw_data = pd.read_csv("engineering_data.csv")
raw_data.head()

The columns are: `record_id` (a number for each reading), `equipment_id` (which machine group, E1 to E6), `load_pct` (how hard it was working, in percent), `ambient_c` (air temperature in degrees C) and `target` (the cooling load in kW). The first two columns are labels. `load_pct` and `ambient_c` are our **features**. `target` is what we predict.

---
## Step 1: Look before you clean

We always look at a table before we change it.

**PREDICT first.** How many rows and columns does the table have? Is there any missing value?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
print("rows, columns:", raw_data.shape)
print()
print("missing values (NaN) in each column:")
print(raw_data.isna().sum())

**READ the output.** Which columns have missing values? A missing value in `target` is a different problem from a missing value in a feature column. Think about what we can do with a row that has no target value. Why is it more serious?

> My answer: 

---
## Step 2: Clean a copy: remove the rows we cannot trust

Three kinds of bad row: a repeated record_id, a row with a missing target value, and a load outside 0 to 100 percent.

**PREDICT first.** How many rows will we lose in total? Use what the slides showed you.

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
modelling_data = raw_data.drop_duplicates("record_id").copy()      # keep the first copy of each record_id
print("after removing repeats:", len(modelling_data))

has_target = modelling_data["target"].notna()                    # True where the target is recorded
load_ok = modelling_data["load_pct"].between(0, 100)             # True where load is between 0 and 100
modelling_data = modelling_data[has_target & load_ok].copy()     # keep rows that pass both checks
print("rows left to model with:", len(modelling_data))

**READ the output.** Does the number of rows left match the number on the "Check your numbers" slide?

> My answer: 

**CHANGE one thing.** Change `between(0, 100)` to `between(0, 90)` and run the cell again. Then change it back to 100 and run it once more. What does this tell you about how much one cleaning rule can change your data?

> What I saw: 

---
## Step 3: Hold out data for a fair evaluation

Equipment groups E1 to E4 form the training set, E5 the validation set and E6 the test set. The validation set and the test set are held-out data: rows not used to fit the model. The test set stays untouched until the end of the course block.

In [ ]:
train_groups = ["E1", "E2", "E3", "E4"]

training_data = modelling_data[modelling_data["equipment_id"].isin(train_groups)].copy()
validation_data = modelling_data[modelling_data["equipment_id"] == "E5"].copy()
test_data = modelling_data[modelling_data["equipment_id"] == "E6"].copy()   # we do not touch this today

feature_columns = ["load_pct", "ambient_c"]
print("training, validation, test rows:", len(training_data), len(validation_data), len(test_data))

**READ the output.** Why do we split by equipment group? What could go wrong if we picked random rows from every group?

> My answer: 

In [ ]:
# the features (X) are a table. The target variable (y) is one column.
X_train = training_data[feature_columns]
y_train = training_data["target"]
X_valid = validation_data[feature_columns]
y_valid = validation_data["target"]

print("X_train shape:", X_train.shape, "  y_train shape:", y_train.shape)
print("missing values in X_train:", int(X_train.isna().sum().sum()), "  in X_valid:", int(X_valid.isna().sum().sum()))

There is one missing value in a feature column, in `ambient_c` in the training set. A model cannot use a missing value, so in Step 5 **imputation** will fill it with the **median** (the middle value) of that column, using `SimpleImputer`. The median is found from the training set only, and the same number is used for any missing value in the validation set.

---
## Step 4: Model 1: the baseline model

A baseline model is the simplest possible model. Here it is a mean predictor (`DummyRegressor`): it predicts the same number for every row, the mean of the training targets (cooling load). Every other model must beat it.

**PREDICT first.** About how far off, in kW, will the baseline model be on the validation set? Give a number.

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

baseline_model = DummyRegressor(strategy="mean")   # create the model
baseline_model.fit(X_train, y_train)               # training (fitting): learn the mean of the training targets
y_pred_baseline = baseline_model.predict(X_valid)   # inference: the same number for every validation row

baseline_mae = mean_absolute_error(y_valid, y_pred_baseline)
print("The baseline model predicts", round(y_pred_baseline[0], 2), "kW for every row.")
print("Mean absolute error:", round(baseline_mae, 2), "kW")

**Check by hand.** The prediction error (residual) of one row is `actual - predicted`, that is e = y - y-hat. The cell below shows five validation rows so that you can see the arithmetic behind the MAE: the MAE is the mean of the `absolute_error` column over all validation rows.

In [ ]:
check_rows = validation_data[["record_id", "target"]].head(5).copy()
check_rows["predicted"] = y_pred_baseline[:5]
check_rows["error"] = check_rows["target"] - check_rows["predicted"]
check_rows["absolute_error"] = check_rows["error"].abs()
check_rows.round(2)

**READ the output.** Pick one row. Check that `error = target - predicted`. Is the error positive when the baseline model predicted too low?

> My answer: 

---
## Step 5: Model 2: multiple linear regression

A linear regression model predicts the target as an intercept plus weighted features. With two features it is called multiple linear regression (with one feature it is simple linear regression). Training chooses the parameters that give the smallest mean squared error on the training set. The formula is y-hat = b0 + b1 x1 + b2 x2, where x1 is `load_pct`, x2 is `ambient_c`, b0 is the intercept (bias) and b1, b2 are the coefficients (weights). We chain the imputer and the model in a scikit-learn **pipeline**, which runs the steps in order.

**PREDICT first.** Will multiple linear regression beat the baseline model? By a little or by a lot?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression

linear_model = make_pipeline(SimpleImputer(strategy="median"), LinearRegression())
linear_model.fit(X_train, y_train)            # training (fitting): learn the median, then b0, b1 and b2
y_pred_linear = linear_model.predict(X_valid)    # inference: predictions for the validation set

linear_mae = mean_absolute_error(y_valid, y_pred_linear)
linear_rmse = mean_squared_error(y_valid, y_pred_linear) ** 0.5
print("Mean absolute error:", round(linear_mae, 2), "kW")
print("Root mean squared error:", round(linear_rmse, 2), "kW")

In [ ]:
# open the model and read the parameters it learned
lin_reg = linear_model.named_steps["linearregression"]
print("intercept:", round(lin_reg.intercept_, 3))
print("coefficient b1 for load_pct:", round(lin_reg.coef_[0], 3))
print("coefficient b2 for ambient_c:", round(lin_reg.coef_[1], 3))

**READ the output.** In plain words, what does the coefficient (weight) b1 for `load_pct` say about cooling load? Complete this sentence and use the unit kW: "When load_pct goes up by 1 percent and ambient_c stays the same, the prediction goes up by ..."

> My answer: 

In [ ]:
# check the formula by hand on the first validation row: b0 + b1 * load_pct + b2 * ambient_c
first_row = validation_data.iloc[0]
by_hand = lin_reg.intercept_ + lin_reg.coef_[0] * first_row["load_pct"] + lin_reg.coef_[1] * first_row["ambient_c"]
print("by hand:", round(by_hand, 2), "kW")
print("model.predict:", round(y_pred_linear[0], 2))

**READ the output.** Are the two numbers the same? What does that tell you about what the model has learned (its parameters)?

> My answer: 

In [ ]:
# plot the actual values against the predicted values. Points near the dashed line are accurate predictions (small prediction errors).
plt.figure(figsize=(5, 4))
plt.scatter(y_valid, y_pred_linear)
low, high = y_valid.min(), y_valid.max()
plt.plot([low, high], [low, high], linestyle="--")
plt.xlabel("actual cooling load (kW)")
plt.ylabel("predicted (kW)")
plt.title("Multiple linear regression, validation set")
plt.show()

**CHANGE one thing.** In Step 3 the features are `["load_pct", "ambient_c"]`. Make a new pipeline that uses only `["load_pct"]` and compare its MAE with the one above. Does the second feature help? This is simple linear regression (one feature) from the slides, and the model above is multiple linear regression.

> What I saw: 

In [ ]:
# your experiment: simple linear regression, one feature only (load_pct)
simple_model = make_pipeline(SimpleImputer(strategy="median"), LinearRegression())
simple_model.fit(training_data[["load_pct"]], y_train)
simple_mae = mean_absolute_error(y_valid, simple_model.predict(validation_data[["load_pct"]]))
print("MAE of simple linear regression (load_pct only):", round(simple_mae, 2), "kW")

---
## Step 6: Model 3: a decision tree

A decision tree asks a short list of yes or no questions about the features (internal nodes) and ends in leaf nodes that give the prediction. `max_depth=4` is a hyperparameter: a setting we choose before training. It allows at most four questions in a row. `random_state=17` makes the result repeatable.

**PREDICT first.** Will the decision tree beat multiple linear regression? Give a reason.

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
from sklearn.tree import DecisionTreeRegressor

tree_model = make_pipeline(SimpleImputer(strategy="median"), DecisionTreeRegressor(max_depth=4, random_state=17))
tree_model.fit(X_train, y_train)
y_pred_tree = tree_model.predict(X_valid)

tree_mae = mean_absolute_error(y_valid, y_pred_tree)
tree_rmse = mean_squared_error(y_valid, y_pred_tree) ** 0.5
print("Mean absolute error:", round(tree_mae, 2), "kW")
print("Root mean squared error:", round(tree_rmse, 2), "kW")

**CHANGE one thing.** Change `max_depth=4` to `max_depth=1`, then `2`, then `8`, running the cell each time and writing down the MAE. Put the depth back to 4 afterwards. Do not worry about the pattern yet: in Class 3 we explain what you are seeing (overfitting and underfitting).

> What I saw: 

---
## Step 7: The model comparison table

All three models were trained on the same training set and evaluated on the same validation set. That is what makes the comparison fair.

In [ ]:
baseline_rmse = mean_squared_error(y_valid, y_pred_baseline) ** 0.5
# MSE is the mean of the squared errors (unit squared). RMSE is its square root (the unit of the target).
mse = [mean_squared_error(y_valid, p) for p in (y_pred_baseline, y_pred_linear, y_pred_tree)]
results = pd.DataFrame({
    "model": ["Baseline model (mean predictor)", "Multiple linear regression", "Decision tree, max_depth 4"],
    "MAE": [baseline_mae, linear_mae, tree_mae],
    "MSE": mse,
    "RMSE": [baseline_rmse, linear_rmse, tree_rmse],
}).round(2)
print(results.to_string(index=False))

In [ ]:
plt.figure(figsize=(6, 3.5))
plt.barh(results["model"], results["MAE"])
plt.xlabel("mean absolute error (kW), lower is better")
plt.gca().invert_yaxis()
plt.show()

**Write your conclusion in two sentences.** Which model is best on the validation set? How much better is it than the baseline model? Use the unit in your answer.

> My conclusion: 

**What this does not prove.** The data are synthetic teaching data. The validation set has only 15 rows. The test set (E6) has not been used. In Classes 3 and 4 we learn how to judge a model with more care.

---
## Step 8: Save your work

Choose **Runtime, then Restart session and run all**. If every cell runs without an error, choose **File, then Save a copy in Drive**. Keep this notebook file safe. Class 2 uses a new notebook, and you reuse the cleaning steps from this one. This notebook is for learning. It is not the graded upload.